In [1]:
import huggingface_hub
import os

In [2]:
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com" 

In [ ]:
#52 files for d12 768
#180 files for d20 1280
for i in range(50):
    try:
        huggingface_hub.snapshot_download(
            repo_id="karpathy/fineweb-edu-100b-shuffle",
            repo_type="dataset",
            local_dir="./data/processed/pretrain/t2t",
            allow_patterns=[f"shard_{i:05d}.parquet" for i in range(52)],
            max_workers=4
        )
    except Exception as e:print("重试:", e)


Fetching ... files: 0it [00:00, ?it/s]

重试: An error happened while trying to locate the file on the Hub and we cannot find the requested files in the local cache. Please check your connection and try again or make sure your Internet connection is on.


Fetching ... files: 0it [00:00, ?it/s]

重试: peer closed connection without sending complete message body (received 60964090 bytes, expected 94277456)


Fetching ... files: 0it [00:00, ?it/s]

...

Fetching ... files: 0it [00:00, ?it/s]

Fetching ... files: 0it [00:00, ?it/s]

Fetching ... files: 0it [00:00, ?it/s]

Fetching ... files: 0it [00:00, ?it/s]

KeyboardInterrupt: 

In [9]:
from model import gpt
from  tokenizer import tokenizer
num_layers=20
P=12
gpt_config=gpt.GPTConfig(n_layer=num_layers)
tokeniz=tokenizer.Tokenizer.from_dict("weight/tokenizer.pkl")
model=gpt.GPT(gpt_config,"cpu",tokeniz)
params_count=model.get_model_scale()
params_count

{'emb_count': 50331648,
 'transformer_matrix': 141557760,
 'lm_head_count': 50331648,
 'total_count': 242221056}

In [ ]:
import time
from torch.utils.data import DataLoader
from tasks import pretraindata
from common import data_paths

seq_len = 1024          # 和 pretrain.py 的 --seq_len 保持一致（截断方式依赖它）
batch_size = 64         # 只影响统计速度，不影响结果
num_workers = 8         # 每个 worker 按 global_rg_idx 分到不同的 row group，互不重复

train_dataset = pretraindata.PretrainDataset(
    data_paths.PRETRAIN_T2T, 'train', tokeniz, batch_size, seq_len, rank=0, world_size=1
)
assert train_dataset.data_files, "没找到 shard_*.parquet"
loader = DataLoader(train_dataset, batch_size=None, num_workers=num_workers)

target_tokens = 0
steps = 0
finished = set()        # 已经读完自己那份数据（epoch 变成 1）的 worker
t0 = time.time()
for x, y, segment, state in loader:
    wid = state['worker']
    # 数据集是无限循环的：某个 worker 的 epoch 变成 1，说明它负责的 row group 都读完了一遍
    if state['epoch'] >= 1:
        finished.add(wid)
        if len(finished) == num_workers:
            break
        continue
    target_tokens += y.numel()
    steps += 1
    if steps % 1000 == 0:
        print(f"step {steps} | file {state['f_idx']}/{len(train_dataset.data_files)} "
              f"| {target_tokens/1e9:.3f}B tokens | {time.time()-t0:.0f}s")

total_params = params_count['transformer_matrix']+params_count['lm_head_count']
print(f"一个 epoch 的 target token 数: {target_tokens:,} ({target_tokens/1e9:.3f}B)")
print(f"相当于 {total_params:,} 参数的 {target_tokens/total_params:.1f} tokens/param")


step 1000 | file 2/29 | 0.066B tokens | 11s
step 2000 | file 4/29 | 0.131B tokens | 19s
step 3000 | file 6/29 | 0.197B tokens | 29s
step 4000 | file 8/29 | 0.262B tokens | 39s
step 5000 | file 10/29 | 0.328B tokens | 47s
step 6000 | file 13/29 | 0.393B tokens | 58s
step 7000 | file 15/29 | 0.459B tokens | 68s
step 8000 | file 17/29 | 0.524B tokens | 77s
step 9000 | file 19/29 | 0.590B tokens | 88s
step 10000 | file 21/29 | 0.655B tokens | 98s
step 11000 | file 23/29 | 0.721B tokens | 108s
step 12000 | file 25/29 | 0.786B tokens | 119s
step 13000 | file 27/29 | 0.852B tokens | 130s
一个 epoch 的 target token 数: 895,549,440 (0.896B)
相当于 185,597,952 参数的 4.8 tokens/param


In [10]:
total_params = params_count['transformer_matrix']+params_count['lm_head_count']
print(f"一个 epoch 的 target token 数: {target_tokens:,} ({target_tokens/1e9:.3f}B)")
print(f"相当于 {total_params:,} 参数的 {target_tokens/total_params:.1f} tokens/param")

一个 epoch 的 target token 数: 895,549,440 (0.896B)
相当于 191,889,408 参数的 4.7 tokens/param
